In [1]:
using Base.Threads
using Plots
using Distributions
using LinearAlgebra
#using StatsPlots
using JSON
using JSON3

In [2]:
function find_velocity(mass)
    # mass in kg
    # from kramer 2010 "allometric scaling of resource acquisition"
    #Consumer Velocity (meters/second)
    velocity = (0.5 * mass^0.13);
    return velocity
end

function bite_size_allo(mass)
    # mass in kg
    # bite size in g
    bite_size = 0.002 * ((mass)^0.969); # g
    return bite_size

end

function number_of_chews(mass)
    # shipley 94
#chew/g (processed to mean particle size (allo) and swallowed)
# mass in kg
    chews_per_gram = 343.71 * (mass^(-0.83)); 
    return chews_per_gram
end

function chew_rate_allo(mass)
    # from "dental functional morphology predicts scaling"
    # mass in kg
    # duration in ms
    chewing_cycle_duration = (228.0* (mass)^0.246) / 1000;
    return 1 / (chewing_cycle_duration )  #[s/chew -> chews/s]

end

function chew_allo(mass)
    # allometric function for mouth->gut rate
    chew_rate = chew_rate_allo(mass); # chews/s
    chews_per_gram = number_of_chews(mass)   # chew/kg
    chew = chew_rate / chews_per_gram        # chew/s / chew/g -> g/s
    return chew
end

function gut_volume_g(mass)
    capacity = (0.030 * (mass)^0.881);
    return capacity * 1000 #[kg -> g]
    #return capacity
end

# function indperarea(mass)
#     #Enter mass in kg
#     #Convert to grams
#     massg= mass*1000;
#     popdensity= (0.0116)*massg^-0.776; #inds/area (from Damuth)
#     return popdensity
# end

function foragingtime(mass)
    #Owen Smith 1988 book (data grabbed using PlotDigitizer in /data/)
    #mass in kg
    #foraging time in % of 24 hours

    forageperc = 21.0885*mass^0.124
    forageperc_upper95 = 27*mass^0.17

    #translate to hours
    foragehrs = (forageperc/100)*24
    foragehrs_upper95 = (forageperc_upper95/100)*24

    return foragehrs, foragehrs_upper95 # hours
end

function bc(yvalue, yc, ymax)
    return clamp(yvalue, yc, ymax)
end

function energy_cost(mass)
    ## energy in kcal
    energy = 0.15*(mass^0.68)
    return energy
end

function mean_energy_closed(mass)
    #enter mass in kg
#massg=mass*1000 #convert to grams
    mean_edensity_closed=30*(mass^-0.24)#kcal
    return mean_edensity_closed
end

function mean_energy_open(mass)
    #enter mass in kg
#massg=mass*1000 #convert to grams
    mean_edensity_open=20*(mass^-0.24)#kcal
    return mean_edensity_open
end

#reaction distance
function reactionwidth(mass)
    #mass in kg
    #distance in meters
    #see 'fit' with Pawar data in analysis
    #Note it is not a fit, but anchoring the intercept 

    width = 2 * 5 * (mass^(1/3)) #meters

    return width

end
# reaction height
function reactionheight(mass)
    #mass in KG
    #distance in meters ~ shoulder height
    #From Larramendi 2016
    ra = 0.1501*(mass^0.357)
    return ra
end

function indperarea(mass)
    # mass in kg
    # Pop density: Damuth 1981
    popdensity = (5.45e-5) * mass^-0.776  # inds/area

    # Foraging bout time (s)
    foragebout_hrs, _ = foragingtime(mass)
    foragebout_s = foragebout_hrs * 3600

    # Foraging velocity (m/s)
    foragevelocity = find_velocity(mass)

    # Area foraged as corridor (m²)
    corridorwidth = 2  # meters
    corridorareaforaged = foragevelocity * foragebout_s * corridorwidth

    # OR: use Owen-Smith homerange model (commented out)
    # HR = 13500 * mass^1.25

    # Total individuals in foraged area
    popinarea = 1.0 + popdensity * corridorareaforaged

    return popdensity, popinarea
end
function expectedlifetime(mass)
    #Calder 1984;
    #mass in kg
    #expected lifetime in years
    explifetime = 5.08*mass^0.35;
    explifetime_days = explifetime *365
    return explifetime_days
end
# --- CLOSED habitat (ajuste baseado nos dados do Serengeti) ---
function predation_mortality_open(mass)
    a, b, ymin, ymax = 14.9738, -6.02, 0.08, 0.95
    #a, b, ymin, ymax = 20.1, -8.0, 0.05, 0.9
    p = 1 / (1 + exp(-(a + b * log10(mass))))
    return ymin + (ymax - ymin) * p   # fração anual (0–1)
end

function prob_pred_open(mass)
    annual = predation_mortality_open(mass)
    base = max(1 - annual, 0.0)
    daily_survival = base^(1/365)
    return 1 - daily_survival
end
# --- multiplicador para closed habitat ---
function multiplier_closed(mass; a=5.0, b=-1.3, low=0.1, high=0.9)
    sig = 1 / (1 + exp(-(a + b * log10(mass))))
    return low + (high - low) * sig
end

# --- CLOSED habitat ---
function predation_mortality_closed(mass; a=5.0, b=-1.3, low=0.1, high=0.9)
    base = predation_mortality_open(mass)
    mult = multiplier_closed(mass; a=a, b=b, low=low, high=high)
    return clamp(base * mult, 0.0, 0.98)
end

function prob_pred_closed(mass; a=5.0, b=-1.3, low=0.1, high=0.9)
    annual = predation_mortality_closed(mass; a=a, b=b, low=low, high=high)
    base = max(1 - annual, 0.0)
    daily_survival = base^(1/365)
    return 1 - daily_survival
end

prob_pred_closed (generic function with 1 method)

In [ ]:
using Distributions, StatsBase, Plots

function distance_verification(mass, zeta_g)
    _, n = indperarea(mass)
    beta   = bite_size_allo(mass)
    width  = reactionwidth(mass)
    height = reactionheight(mass)

    mu_g    = 0.001
    alpha_g = 4

    na = n / (width * height)

    m_res_g     = mu_g / beta
    mprime_g    = m_res_g / na
    alphaprime  = alpha_g * na^(zeta_g - 2)
    gammadist   = Gamma(alphaprime, mprime_g / alphaprime)

    gamma_sample = rand(gammadist)
    distance_to_resource = rand(Exponential(1.0 / gamma_sample))

    return distance_to_resource
end


function simulate_stats(masses, zetas, nrep)
    results = Dict()
    for z in zetas
        means = Float64[]
        stds  = Float64[]
        for m in masses
            distances = [distance_verification(m, z) for _ in 1:nrep]
            push!(means, mean(distances))
            push!(stds, std(distances))
        end
        results[z] = (means, stds)
    end
    return results
end


# parâmetros
masses = 1:10:8000
zetas = [1.0, 2.0, 2.15, 2.2]
nrep = 5000  # pode ajustar pra rodar mais rápido

results = simulate_stats(masses, zetas, nrep)


# ---- Plot ----
default(markerstrokewidth=0, legend=:topleft)

p1 = plot(xscale=:log10, xlabel="Body mass (kg)", ylabel="Mean distance (m)", ylims=(0,400))   # <-- limite de y no gráfico da média
for (i, z) in enumerate(zetas)
    means, _ = results[z]
    scatter!(p1, masses, means, label="ζ = $(z)", markersize=4)
end

p2 = plot(xscale=:log10, yscale=:log10, xlabel="Body mass (kg)", ylabel="Std distance (m)")
          #ylim=(1,200))  # <-- limite de y no gráfico do desvio-padrão
for (i, z) in enumerate(zetas)
    _, stds = results[z]
    scatter!(p2, masses, stds, label="ζ = $(z)", markersize=4)
end

plot(p1, p2, layout=(1,2), size=(1000,400))


In [3]:
##com reaction plane
function dailysim(mass, patch, fisio)
    ### 1 = closed habitat
    ### 2 = open habitat
    velocity = find_velocity(mass) # Velocidade do forrageador em m/s
    beta = bite_size_allo(mass)

    if fisio == "grazers" && patch == 1
        beta = 0.5*beta # Ajuste para habitat fechado
    end
    if patch == 1 
        velocity = 0.5*velocity # Ajuste para habitat fechado
    end

    kcal_final = 0.0
    #active_hours = foraging_hours(mass)
    tmax_bout, _ = foragingtime(mass) .* (60 * 60)
    _, n = indperarea(mass)
    chewrate = chew_allo(mass) # g/s
    tchewgram = 1 / chewrate # s/g
    tchew = tchewgram * beta # s/g * g/bite = s/bite
    width = reactionwidth(mass)
    height = reactionheight(mass)
    b0_bmr = 3.4 # Taxa metabólica basal constante (watts kg^-0.75)
    b0_fmr = 6.8 # Taxa metabólica de campo constante (watts kg^-0.75)
    basal_mr = b0_bmr * (mass ^ 0.75) # Taxa metabólica basal para kg
    field_mr = b0_fmr * (mass ^ 0.75) # Taxa metabólica ativa para kg
    upper = 18.0 #maior e menores valores de x na normal
    lower = 0.0 #maior e menores valores de x na normal

    t_travel = 0.0
    t_chew = 0.0
    bites = 0.0
    GUT = 0.0

    total_kcal = 0.0
    cost_ws = 0.0
    number_of_successes = 0.0
    t = 0.0
    gut = 0.0

    max_gut = gut_volume_g(mass)

    if fisio == "mixed"
        #res_traits p/ grazers
        res_traits_g=  [mu=0.001 alpha= 4.5 mean_edensity = mean_energy_closed(mass) sd_edensity = 2.8 zeta= 2.2 d_pred = 0.002;
                        mu= 0.001 alpha = 9 mean_edensity = mean_energy_open(mass) sd_edensity = 1 zeta = 1 d_pred = 0.002;];

        mu_g = res_traits_g[patch,1]
        alpha_g= res_traits_g[patch, 2]
        mean_edensity_g = res_traits_g[patch, 3]
        sd_edensity_g = res_traits_g[patch, 4]
        zeta_g = res_traits_g[patch, 5]

        #res_traits p/ browser
        res_traits_b=  [mu=0.001 alpha= 9 mean_edensity = mean_energy_closed(mass) sd_edensity = 2.8 zeta= 1 d_pred = 0.002;
                        mu= 0.001 alpha = 4.5 mean_edensity = mean_energy_open(mass) sd_edensity = 1 zeta = 2.2 d_pred = 0.002;];               
        
        mu_b = res_traits_b[patch, 1]
        alpha_b = res_traits_b[patch, 2]
        mean_edensity_b = res_traits_b[patch, 3]
        sd_edensity_b = res_traits_b[patch, 4]
        zeta_b = res_traits_b[patch, 5]
        d_pred = res_traits_b[patch,6]
        #edensity = Normal(mean_edensity, sd_edensity)
        #Reaction plane area
        
        if patch == 2 && height>2
            height = 2# Ajuste para habitat aberto
        end
                        #Competitor density (per reaction plane area)
        na = n/(width*height);
                        
                        # Bite density field
        m_res_b = mu_b * (1 / beta) #* height #* width
        m_res_g = mu_g *(1/beta)

                        #Rescaling bite density by Competitor Load
        mprime_b = m_res_b / na #m para browser
        mprime_g = m_res_g / na #m para grazers

        alphaprime_b = alpha_b * na^(zeta_b - 2) #alpha para browsers
        alphaprime_g = alpha_g * na^(zeta_g - 2) #alpha para browsers

        
        edensity_b = Truncated(Normal(mean_edensity_b, sd_edensity_b), lower, upper)
        edensity_g = Truncated(Normal(mean_edensity_g, sd_edensity_g), lower, upper)

        # Inicialização de variáveis escalares


        gammadist_b = Gamma(alphaprime_b, mprime_b / alphaprime_b)
        gammadist_g = Gamma(alphaprime_g, mprime_g / alphaprime_g)

        while t < tmax_bout
            distance_to_resource_b = rand(Exponential(1.0 / rand(gammadist_b)))
            distance_to_resource_g = rand(Exponential(1.0 / rand(gammadist_g)))

            deltat_b = distance_to_resource_b / velocity
            cost_ws_b = field_mr * deltat_b
            cost_kcal_b = cost_ws_b*0.0002

            deltat_g = distance_to_resource_g / velocity
            cost_ws_g = field_mr * deltat_g
            cost_kcal_g = cost_ws_g*0.0002
            # Atualiza tempo e custos
            kcal_food_b = rand(edensity_b)
            kcal_food_g = rand(edensity_g)

            if (kcal_food_b-cost_kcal_b)>(kcal_food_g-cost_kcal_g)

                deltat = deltat_b
                t += deltat
                t_travel += deltat
                cost_ws += field_mr * deltat

                if (tmax_bout > (t + deltat)) && (max_gut > gut)
                    number_of_successes += 1
                    
                    # tb = total_biomass(mass, res_traits, patch)
                    # if Int64(round(tb/ bite_size_allo(mass))) < 0
                    #     println("negative total biomass")
                    # end
                    #try
                    bites += 1
                    #catch e 
                    #    display("deu erro", e)
                    #end
                    
                    t += tchew
                    t_chew += tchew 
                    gut += beta
                    cost_ws += field_mr * tchew + d_pred
                    GUT = gut
                    kcal_food = kcal_food_b
                    kcal_final += beta * kcal_food
                else
                    break
                end
            else
                deltat = deltat_g
                t += deltat
                t_travel += deltat
                cost_ws += field_mr * deltat

                if (tmax_bout > (t + deltat)) && (max_gut > gut)
                    number_of_successes += 1
                    
                    # tb = total_biomass(mass, res_traits, patch)
                    # if Int64(round(tb/ bite_size_allo(mass))) < 0
                    #     println("negative total biomass")
                    # end
                    #try
                    bites += 1
                    #catch e 
                    #    display("deu erro", e)
                    #end
                    
                    t += tchew
                    t_chew += tchew 
                    gut += beta
                    cost_ws += field_mr * tchew + d_pred
                    GUT = gut
                    kcal_food = kcal_food_g
                    kcal_final += beta * kcal_food
                else
                    break
                end
            end
        end

    else
        if fisio == "grazers" 
            res_traits= [mu=0.001 alpha= 4.5 mean_edensity = mean_energy_closed(mass) sd_edensity = 2.8 zeta= 2.2 d_pred = 0.002;
                        mu= 0.001 alpha = 9 mean_edensity = mean_energy_open(mass) sd_edensity = 1 zeta = 1 d_pred = 0.002;];
        
        
        elseif fisio == "browsers" 
            res_traits= [mu= 0.001 alpha= 9 mean_edensity = mean_energy_closed(mass) sd_edensity = 2.8 zeta= 1 d_pred = 0.002;
                        mu= 0.001 alpha = 4.5 mean_edensity = mean_energy_open(mass) sd_edensity = 1 zeta = 2.2 d_pred = 0.002;];
        
        end

        # Mover as definições para antes do uso
        mu = res_traits[patch, 1]
        alpha = res_traits[patch, 2]
        mean_edensity = res_traits[patch, 3]
        sd_edensity = res_traits[patch, 4]
        zeta = res_traits[patch, 5]
        d_pred = res_traits[patch,6]
        #edensity = Normal(mean_edensity, sd_edensity)
        #Reaction plane area
        width = reactionwidth(mass)
        height = reactionheight(mass)

        if patch == 2 && height>2
            height = 2# Ajuste para habitat aberto
        end
                        #Competitor density (per reaction plane area)
        na = n/(width*height);
                        
                        # Bite density field
        m_res = mu * (1 / beta) #* height #* width

                        #Rescaling bite density by Competitor Load
        mprime = m_res / na

        alphaprime = alpha * na^(zeta - 2)
        # WATTS/KG
        edensity = Truncated(Normal(mean_edensity, sd_edensity), lower, upper)
        gammadist = Gamma(alphaprime, mprime / alphaprime)
        

        while t < tmax_bout
            distance_to_resource = rand(Exponential(1.0 / rand(gammadist)))

            # Atualiza tempo e custos
            deltat = distance_to_resource / velocity
            t += deltat
            t_travel += deltat
            cost_ws += field_mr * deltat

            if (tmax_bout > (t + deltat)) && (max_gut > gut)
                number_of_successes += 1
                
                # tb = total_biomass(mass, res_traits, patch)
                # if Int64(round(tb/ bite_size_allo(mass))) < 0
                #     println("negative total biomass")
                # end
                #try
                bites += 1
                #catch e 
                #    display("deu erro", e)
                #end
                
                t += tchew
                t_chew += tchew 
                gut += beta
                cost_ws += field_mr * tchew + d_pred
                GUT = gut
                kcal_food = rand(edensity)
                kcal_final += beta * kcal_food
            else
                break
            end
        end
    end
    total_t = 60 * 60 * 24
    rest_t = total_t - t
    cost_ws += basal_mr * rest_t
    cost_whr = cost_ws / 3600 # Convertendo watt/s para watt/hr
    cost_kcal = cost_whr * 0.86 # Convertendo watt para kcal
    total_kcal = kcal_final

    return total_kcal, number_of_successes, cost_kcal
end

dailysim (generic function with 1 method)

In [ ]:
using Plots

# --- multiplicador para closed habitat ---
function multiplier_closed(mass; a=5, b=-1.3, low=0.3, high=0.95)
    sig = 1 / (1 + exp(-(a + b * log10(mass))))
    return low + (high - low) * sig
end

# --- valores de massa ---
masses = collect(1.0:1.0:50000.0)  # massas de 1 kg a 5000 kg

# --- curva do multiplicador ---
mult_curve = [multiplier_closed(m) for m in masses]

# --- plot ---
plot(masses, mult_curve, lw=2, color=:blue, label="Multiplicador closed",
     xlabel="Mass (kg)", ylabel="Multiplier")
Plots.hline!([1.0], ls=:dash, color=:black, label="=1 (no efect)")


In [ ]:
using Plots

# --- Funções (iguais às que você já tem) ---
function expectedlifetime(mass)
    explifetime = 5.08*mass^0.35
    return explifetime * 365
end

function predation_mortality_open(mass)
    a, b, ymin, ymax = 14.9738, -6.02, 0.05, 0.95
    p = 1 / (1 + exp(-(a + b * log10(mass))))
    return ymin + (ymax - ymin) * p   # fração anual (0–1)
end

function prob_pred_open(mass)
    annual = predation_mortality_open(mass)
    base = max(1 - annual, 0.0)
    daily_survival = base^(1/365)
    return 1 - daily_survival
end

function multiplier_closed(mass; a=5.0, b=-1.3, low=0.1, high=1.3)
    sig = 1 / (1 + exp(-(a + b * log10(mass))))
    return low + (high - low) * sig
end

function predation_mortality_closed(mass; a=5.0, b=-1.3, low=0.1, high=1.3)
    base = predation_mortality_open(mass)
    mult = multiplier_closed(mass; a=a, b=b, low=low, high=high)
    return clamp(base * mult, 0.0, 0.98)
end

function prob_pred_closed(mass; a=5.0, b=-1.3, low=0.1, high=1.3)
    annual = predation_mortality_closed(mass; a=a, b=b, low=low, high=high)
    base = max(1 - annual, 0.0)
    daily_survival = base^(1/365)
    return 1 - daily_survival
end

# ----------------------------
# Plotando
# ----------------------------
masses = collect(1.0:1.0:10000.0)  # massas de 1 kg a 5000 kg

probs_open = [prob_pred_open(m) for m in masses]
probs_closed = [prob_pred_closed(m) for m in masses]

annual_open = [predation_mortality_open(m) for m in masses]
annual_closed = [predation_mortality_closed(m) for m in masses]

# Plot 1: probabilidade diária
# p1 = plot(masses, probs_open, label="Open habitat", lw=2, color=:blue)
# plot!(p1, masses, probs_closed, label="Closed habitat", lw=2, color=:red, ls=:dash, xaxis=:log10)
# xlabel!(p1, "Body mass (kg)")
# ylabel!(p1, "Daily predation probability")
# title!(p1, "Daily predation risk")
default(
    fontfamily = "Helvetica",
    guidefont = font(18),
    tickfont = font(15),
    legendfont = font(13),
    dpi = 600
)
# Plot 2: mortalidade anual
p2 = plot(masses, annual_open, lw=5, color=:blue, ylims=(0,1),size = (800, 650), grid = false, framestyle = :box )
plot!(p2, masses, annual_closed, lw=5,color=:red, ls=:dash, xaxis=:log10, legend=:none)
xlabel!(p2, "Body mass (kg)")
ylabel!(p2, "Annual predation mortality")
#title!(p2, "Lifetime predation risk per mass")

# Mostrar os dois juntos
#plot(p1, p2, layout=(1,2), size=(1000,400))
plot(p2)
savefig("prob_predation_closed.svg")

In [6]:
function calculate_histogram(data, nbins)
    # Define bin edges from minimum to maximum, with nbins intervals
    min_val = minimum(data)
    max_val = maximum(data)
    bin_edges = range(min_val, max_val, length=nbins+1)
    bin_counts = zeros(Int, nbins)

    # Bin data into counts
    for val in data
        for i = 1:nbins
            if val >= bin_edges[i] && val < bin_edges[i+1]
                bin_counts[i] += 1
                break
            end
        end
    end

    # Edge case: include data that matches the maximum value
    if data[end] == max_val
        bin_counts[end] += 1
    end

    return bin_edges, bin_counts
end

function find_bin_index(value, bins)
    nbins = length(bins) - 1
    # Loop through the bins to find where the value fits.
    for i = 1:nbins
        if value >= bins[i] && value < bins[i+1]
            return i
        end
    end
    # If the value is exactly equal to the maximum, assign it to the last bin.
    return nbins
end

find_bin_index (generic function with 1 method)

In [7]:
mass_dict = Dict()
habitat = collect(1:2)
#mass[strat] = Dict()
configurations =20000
nbins=50
mass= 1000
#for mass in collect(10:100:2110) # Masses in kg  
    # Initialize a dictionary for each mass. 
mass_dict[mass] = Dict()
for patch in 1:length(habitat) 
    mass_dict[mass][patch] = Dict()
    gains = Float64[]
    costs = Float64[]
    #gut = Float64[]
    for _ in 1:configurations
        # Call the dailysim function to simulate one round of foraging and collect the total kilojoules gained and the cost.
        total_mcal, _, cost_mcal = dailysim(mass, patch, "mixed") # Use "even" for both closed and open patchesrs
        # Store the results in the gains and costs vectors.
        push!(gains, total_mcal)
        push!(costs, cost_mcal)
        #push!(gut, GUT)
    end

    gains_bin_edges, gains_bin_counts = calculate_histogram(gains, nbins)
    costs_bin_edges, costs_bin_counts = calculate_histogram(costs, nbins)
    #gut_bin_edges, gut_bin_counts = calculate_histogram(gut, nbins)
    gains_probabilities = gains_bin_counts / sum(gains_bin_counts)
    costs_probabilities = costs_bin_counts / sum(costs_bin_counts)
    #gut_probabilities = gut_bin_counts / sum(gut_bin_counts)

    # Calculate the midpoints of bins for gains and costs for better representation.
    gains_bin_midpoints = (gains_bin_edges[1:end-1] + gains_bin_edges[2:end]) / 2;
    costs_bin_midpoints = (costs_bin_edges[1:end-1] + costs_bin_edges[2:end]) / 2;
    #gut_bin_midpoints = (gut_bin_edges[1:end-1] + gut_bin_edges[2:end])/2

    # Pair up each gain with its corresponding cost.
    gain_cost_pairs = [(gains[i], costs[i]) for i in 1:length(gains)]
    
    # Define bins for the joint distribution of gains and costs.
    gain_bins = range(minimum(gains), maximum(gains), length=nbins)
    cost_bins = range(minimum(costs), maximum(costs), length=nbins)
    #gut_bins = range(minimum(gut), maximum(gut), length = nbins)
    
    # Initialize a matrix to calculate the joint histogram of gains and costs.
    joint_histogram = zeros(Int, nbins, nbins)
    
    # Fill in the joint histogram by finding the appropriate bin for each gain-cost pair.
    for pair in gain_cost_pairs
        gain_bin_index = find_bin_index(pair[1], gain_bins)
        cost_bin_index = find_bin_index(pair[2], cost_bins)
        #gut_bin_index = find_bin_index(pair[3], gut_bins)
        joint_histogram[gain_bin_index, cost_bin_index] += 1
    end
    
    # Convert the counts in the joint histogram to probabilities.
    joint_probabilities = joint_histogram / sum(joint_histogram)
    mass_dict[mass][patch]["gains"]= gain_bins
    mass_dict[mass][patch]["costs"]= cost_bins
    #mass_dict[mass][patch]["gut"]= gut_bins
    mass_dict[mass][patch]["prob"]= joint_probabilities
end
#end


In [ ]:
mass_dict = Dict()
habitat = collect(1:2)
#mass[strat] = Dict()
configurations =20000
nbins=50
for mass in collect(10:100:2110) # Masses in kg  
    # Initialize a dictionary for each mass. 
    mass_dict[mass] = Dict()
    for patch in 1:length(habitat) 
        mass_dict[mass][patch] = Dict()
        gains = Float64[]
        costs = Float64[]
        #gut = Float64[]
        for _ in 1:configurations
            # Call the dailysim function to simulate one round of foraging and collect the total kilojoules gained and the cost.
            total_mcal, _, cost_mcal = dailysim(mass, patch, "browsers") # Use "even" for both closed and open patchesrs
            # Store the results in the gains and costs vectors.
            push!(gains, total_mcal)
            push!(costs, cost_mcal)
            #push!(gut, GUT)
        end

        gains_bin_edges, gains_bin_counts = calculate_histogram(gains, nbins)
        costs_bin_edges, costs_bin_counts = calculate_histogram(costs, nbins)
        #gut_bin_edges, gut_bin_counts = calculate_histogram(gut, nbins)
        gains_probabilities = gains_bin_counts / sum(gains_bin_counts)
        costs_probabilities = costs_bin_counts / sum(costs_bin_counts)
        #gut_probabilities = gut_bin_counts / sum(gut_bin_counts)

        # Calculate the midpoints of bins for gains and costs for better representation.
        gains_bin_midpoints = (gains_bin_edges[1:end-1] + gains_bin_edges[2:end]) / 2;
        costs_bin_midpoints = (costs_bin_edges[1:end-1] + costs_bin_edges[2:end]) / 2;
        #gut_bin_midpoints = (gut_bin_edges[1:end-1] + gut_bin_edges[2:end])/2

        # Pair up each gain with its corresponding cost.
        gain_cost_pairs = [(gains[i], costs[i]) for i in 1:length(gains)]
        
        # Define bins for the joint distribution of gains and costs.
        gain_bins = range(minimum(gains), maximum(gains), length=nbins)
        cost_bins = range(minimum(costs), maximum(costs), length=nbins)
        #gut_bins = range(minimum(gut), maximum(gut), length = nbins)
        
        # Initialize a matrix to calculate the joint histogram of gains and costs.
        joint_histogram = zeros(Int, nbins, nbins)
        
        # Fill in the joint histogram by finding the appropriate bin for each gain-cost pair.
        for pair in gain_cost_pairs
            gain_bin_index = find_bin_index(pair[1], gain_bins)
            cost_bin_index = find_bin_index(pair[2], cost_bins)
            #gut_bin_index = find_bin_index(pair[3], gut_bins)
            joint_histogram[gain_bin_index, cost_bin_index] += 1
        end
        
        # Convert the counts in the joint histogram to probabilities.
        joint_probabilities = joint_histogram / sum(joint_histogram)
        mass_dict[mass][patch]["gains"]= gain_bins
        mass_dict[mass][patch]["costs"]= cost_bins
        #mass_dict[mass][patch]["gut"]= gut_bins
        mass_dict[mass][patch]["prob"]= joint_probabilities
    end
end


In [ ]:
collect(mass_dict[1010][2]["costs"])

In [8]:
function interpolate(yvalue, S, patch, t, mass)
    lowy = floor(Int, yvalue)
    highy = lowy + 1
    qy = yvalue - lowy

    # Garantir que lowy e highy estejam dentro dos limites
    lowy = max(1, min(lowy, size(S[mass], 2)))
    highy = max(1, min(highy, size(S[mass], 2)))

    S_low = S[mass][patch, lowy, t+1]
    S_high = S[mass][patch, highy, t+1]

    if lowy == highy
        Svalue = S_low
    else
        Svalue = qy * S_high + (1 - qy) * S_low
    end

    return Svalue
end


interpolate (generic function with 1 method)

In [9]:
function runSDP(mass,mass_dict,habitat, max_mcal)
    #println("test")
    #d= [0.09, 0.01] #probabilidade de morte
    tmax=30
    #d= [0.012, 0.011] #probabilidade de morte
    d = [prob_pred_closed(mass), prob_pred_open(mass)] #probabilidade de morte
    #g_bins= zeros(hmax)
    gutrate = 0.6
    #Traveling to patches
    #Number of patches
    hmax = 2;

    # for a in 1:hmax
    #     mass_dict["$mass"]["$a"]["prob"] = reduce(vcat,transpose.(mass_dict["$mass"]["$a"]["prob"]))
    # end
    #PatchTree matrix
    #Defines travel options based on current patch
    #Current patch identified by row, and in column 1
    #Decision 1 = stay in h
    #Decision 2...hmax = go to other patch
    #Note: Alternative patches always in ascending order
    patchtree = Array{Int64}(undef,hmax,hmax);
    for i=1:hmax
        patchtree[i,1] = i;
        patchtree[i,2:hmax] = setdiff(collect(1:hmax),i);
    end
    numpathways = size(patchtree)[2];
    #println("numpathways", numpathways)
    #NOTE: Energetic travel costs
    #Currently binary - need to fill in with actually MegaCal costs
    #Organized as a square adjacency matrix... (different than the patchtree matrix)
    #Row defines current patch
    #Column defines new patch
    #Diagonals are zero bc there is no interpatch travel
    travelcosts = zeros(Float64,hmax,hmax) .+ energy_cost(mass) ;
    travelcosts[diagind(travelcosts)] .= 0;
    starving_bins = zeros(Int, hmax)


    #println("tmax:", tmax)

    f_bins=50 #numero de bins 
    g_bins = 5
    ymax = 4870*0.02*(mass^1.19) # reserva de gordura  em Kcal
    yc = 4870*0.1*0.02*(mass^1.19) # reserva de gordura mínima (Kcal) antes da morte por desnutrição

    #println("ymax:", ymax)

    S[mass] = zeros(hmax, f_bins, tmax)
    T[mass] = zeros(hmax, f_bins,tmax)
    #println("S size:", size(S[mass]))
    fat_bin_size = ymax/f_bins 
    gut_bin_size = (gut_volume_g(mass)*max_mcal)/g_bins#fat bin size in Mcal
    #println("fat_bin_size", fat_bin_size)
    # gut_bin_size = gut_capacity_g(mass)*
    #terminal survival depends linear of the body fat
    a = 1/(ymax-yc)
    b = -yc/(ymax-yc)
    #Across patch states
    for h in 1:hmax
        #Across fat states
        for y in 1:f_bins
            fat_energy = y*fat_bin_size
            if fat_energy>yc
                S[mass][h,y,tmax] = a*fat_energy + b
            else
                # below the minimum fat reserves
                S[mass][h,y,tmax] = 0
            end
        end
        #println( S[mass][:,tmax])
        #NOTE: is this being used?
        starving_bins = sum(S[mass][h,:, tmax] .== 0.0) #know what bins contain the amount of fat below starvation
    end
    


    nbins=length(mass_dict[mass][1]["costs"])

    #println("inicio SDP")
    #println("nbins", nbins)
    for t in (tmax-1):-1:1
        #println("tempo:", t)

        #Cycle through home-patch states
        #h represents the patch that the organism is currently in
        #println("tempo", t)
        for h in 1:hmax
            #println("h", h)
            #Cycle through fat states
           

            for y in (starving_bins + 1):f_bins #NOTE: only cycle through bins where it is alive - otherwise will stay zero
                surv = zeros(numpathways)
                
                for k in 1:numpathways
                    #println("y", y)
                    patch = patchtree[h,k];
                    #println("patch:", patch)
                    travel = travelcosts[h,patch];
                    #Cycle through decisions
                    
                    #Old decision cycle - targeting
                    # for k in 1:length(tid)
                    st=0.0
                    #New decision cycle - patch state
                    for g in 1:g_bins
                        #println("k", k)
                        # k = 1 => Stay in current patch (h); No moving cost
                        # k > 1 => Move from current patch (h) to patch k defined in patchtree; There should be a moving cost
                         #NOTE: convert to megacal

                        
                        #println("k", k)
                        for i in 1:nbins
                            for j in 1:nbins
                            
                                fat_state = y*fat_bin_size #Mcal
                                gut_state = g*gut_bin_size
                                #println("fat state:", fat_state)
                                #NOTE: this assumes that there are different gain/loss distributions associated with PATCH with index $patch
                                gain = mass_dict[mass][patch]["gains"][i] #Mcal
                                #println("gain:", gain)
                                loss = mass_dict[mass][patch]["costs"][j] #Mcal
                                #println("loss:", loss)
                                #gut = mass_dict["$mass"]["$patch"]["gut"][g]

                                prob = mass_dict[mass][patch]["prob"][i,j] #prob
                                #println("prob:", prob)
                                gutpass = (gain+gut_state)*gutrate#considerando por todos os valores possíveis
                                #poo = (1 - gutrate)*gut_state
                                yp = fat_state + gutpass - loss - travel #Mcal

                                #println("yp:", yp)
                                
                                # yp_bin = Int(round(yp/fat_bin_size)) #Mcal to bins
                                yp_bin = yp/fat_bin_size
                                yp_bin = bc(yp_bin, 1, f_bins)

                                # st+= prob*S[mass][yp_bin,t+1]
                                #println("yp_bin", yp_bin)
                                #NOTE: INTERPOLATE
                                Svalue = interpolate(yp_bin, S, patch, t, mass);
                                #println("Svalue", Svalue)
                                st += prob*Svalue*(1/g_bins);
                                
                            end
                        end
                        surv[k] = (1-d[h])*st
                        if surv[k] < 0
                            surv[k] = 0
                        end
                    end
                    maxvalue = maximum(surv)
                    max_indices = findall(x -> x == maxvalue, surv)
                    #println("maxvalue:", maxvalue)
                    if length(max_indices) > 1
                        @warn("Duplicate maximum value found in 'surv' at time $t and state $y")
                        println("surv:", surv)
                    end

                    # Seleciona aleatoriamente uma das ações ótimas
                    strat = rand(max_indices)

                    # Atualiza S e T com o valor máximo e a estratégia selecionada
                    S[mass][h, y, t] = maxvalue
                    #println("S[$mass][$h,$y,$t] = $maxvalue")
                    T[mass][h, y, t] = strat
                end #g loop
            end #y loop
        end #h loop
    end

    return S, T
end

runSDP (generic function with 1 method)

In [ ]:
function runSDP(mass, mass_dict, habitat, max_mcal)
    tmax = 30
    d = [prob_pred_closed(mass), prob_pred_open(mass)] # prob de morte
    gutrate = 0.6

    f_bins = 50 
    g_bins = 5
    ymax = 4870*0.02*(mass^1.19) 
    yc   = 4870*0.1*0.02*(mass^1.19)

    fat_bin_size = ymax / f_bins 
    gut_bin_size = (gut_volume_g(mass) * max_mcal) / g_bins

    a = 1/(ymax - yc)
    b = -yc/(ymax - yc)

    # S = Dict{Int, Dict{Int, Array{Float64,2}}}()
    # T = Dict{Int, Dict}()

    S[mass] = Dict()
    T[mass] = Dict()

    for patch in 1:2
        S[mass][patch]=zeros(f_bins, tmax)

        for y in 1:f_bins
            fat_energy = y * fat_bin_size
            if fat_energy > yc
                S[mass][patch][y, tmax] = a*fat_energy + b
            else
                S[mass][patch][y, tmax] = 0
            end
        end
    end

    nbins = length(mass_dict[mass][1]["costs"])

    for patch in 1:2
        starving_bins = sum(S[mass][patch][:, tmax] .== 0.0)
        for t in (tmax-1):-1:1
            for y in (starving_bins+1):f_bins
                st = 0.0
                for g in 1:g_bins
                    for i in 1:nbins
                        for j in 1:nbins
                            fat_state = y*fat_bin_size
                            gut_state = g*gut_bin_size
                            gain = mass_dict[mass][patch]["gains"][i]
                            loss = mass_dict[mass][patch]["costs"][j]
                            prob = mass_dict[mass][patch]["prob"][i,j]

                            gutpass = (gain+gut_state) * gutrate
                            yp = fat_state + gutpass - loss
                            yp_bin = bc(yp/fat_bin_size, 1, f_bins)

                            Svalue = interpolate(yp_bin, S, patch, t, mass)
                            st += prob * Svalue * (1/g_bins)
                        end
                    end
                end
                S[mass][patch][y,t] = max((1 - d[patch]) * st, 0)
            end
        end
    end

    return S, T
end



In [10]:
S = Dict()
T = Dict()
max_kcal = 3
mass = 1000
S, T = runSDP(mass, mass_dict, habitat, max_kcal)


(Dict{Any, Any}(1000 => [0.0 0.0 … 0.9931714959167659 0.9931714959167659; 0.0 0.0 … 0.9930644966105957 0.9930644966105957;;; 0.0 0.0 … 0.993406183523622 0.993406183523622; 0.0 0.0 … 0.9932991589333884 0.9932991589333884;;; 0.0 0.0 … 0.9936409265874651 0.9936409265874651; 0.0 0.0 … 0.9935338767071934 0.9935338767071934;;; … ;;; 0.0 0.0 … 0.9995275650809485 0.9995275650809485; 0.0 0.0 … 0.9994198810038266 0.9994198810038266;;; 0.0 0.0 … 0.9997637546345248 0.9997637546345248; 0.0 0.0 … 0.9996560451115273 0.9996560451115273;;; 0.0 0.0 … 0.9777777777777776 0.9999999999999998; 0.0 0.0 … 0.9777777777777776 0.9999999999999998]), Dict{Any, Any}(1000 => [0.0 0.0 … 1.0 1.0; 0.0 0.0 … 2.0 2.0;;; 0.0 0.0 … 1.0 1.0; 0.0 0.0 … 2.0 2.0;;; 0.0 0.0 … 1.0 1.0; 0.0 0.0 … 2.0 2.0;;; … ;;; 0.0 0.0 … 1.0 1.0; 0.0 0.0 … 2.0 2.0;;; 0.0 0.0 … 2.0 2.0; 0.0 0.0 … 1.0 1.0;;; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]))

In [ ]:
means_browsers = Dict()

for mass in collect(10:100:2110)
    means_browsers[mass] = Dict()
    for patch in 1:2
        mat = S[mass][patch]
        means_browsers[mass][patch] = mean(mat)   # média de todos os valores
    end
end

In [ ]:
means_grazers = Dict()

for mass in collect(10:100:2110)
    means_grazers[mass] = Dict()
    for patch in 1:2
        mat = S[mass][patch]
        means_grazers[mass][patch] = mean(mat)   # média de todos os valores
    end
end

In [ ]:
using Plots

masses = collect(10:100:2110)

# separar por patch
grazers_patch1 = [means_grazers[m][1] for m in masses]
grazers_patch2 = [means_grazers[m][2] for m in masses]
browsers_patch1 = [means_browsers[m][1] for m in masses]
browsers_patch2 = [means_browsers[m][2] for m in masses]

# plot
plot(masses, grazers_patch1, label="Grazers - Closed Habitat ", lw=2, ylims = (0,0.5), legend=:outerright, size=(700,400) )
plot!(masses, grazers_patch2, label="Grazers - Open Habitat", lw=2)
plot!(masses, browsers_patch1, label="Browsers - Closed Habitat", lw=2, ls=:dash)
plot!(masses, browsers_patch2, label="Browsers - Open Habitat", lw=2, ls=:dash)

xlabel!("Mass (kg)")
ylabel!("Mean fitness")
title!("Mean fitness")
#savefig("mean_fitness_0.0001.png")


In [ ]:
using Statistics, Plots

function mean_fitness_by_mu(S; y_range=25:30)
    results = Dict{Int, Dict{Float64, Float64}}()

    for patch in keys(S)
        results[patch] = Dict{Float64, Float64}()
        for mu in keys(S[patch])
            sub = S[patch][mu][y_range, :]      # fat bins 25–30, todos os tempos
            avg = mean(sub)                     # média em y e t
            results[patch][mu] = avg
        end
    end

    return results
end

function plot_mean_fitness(S; y_range=25:30)
    res = mean_fitness_by_mu(S; y_range=y_range)

    plt = plot(xaxis=:log, title= "Grazers", xlabel="μ", ylabel="mean fitness", legend=:topleft)

    for patch in sort(collect(keys(res)))
        mus = sort(collect(keys(res[patch])))
        vals = [res[patch][mu] for mu in mus]
        plot!(plt, mus, vals, marker=:circle, lw=2, label="Habitat $patch")
    end

    display(plt)
end


In [ ]:
plot_mean_fitness(S, y_range=25:30)


In [ ]:
plot_mean_fitness(S, y_range=25:30)


In [12]:
import Pkg; Pkg.add("Colors")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
    Updating `~/.julia/environments/v1.11/Project.toml`
  [5ae59095] + Colors v0.13.1
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [14]:
using StatsPlots, Colors

mass = 1000
h = 1
tmax = 30
f_bins = 50

# slice - ajuste a transposição conforme o teu formato
S_slice = S[mass][h, :, :] 
S_slice_transposed = S_slice'
default(
    fontfamily = "Helvetica",
    guidefont = font(18),
    tickfont = font(15),
    legendfont = font(13),
    dpi = 600
)
heatmap(
    1:tmax,
    1:f_bins,
    S_slice,
    xlabel = "Time (days)",
    ylabel = "Fat state (kcal)",
    #title = "Survival probability over time – Browsers",
    c = :thermal,
    size = (800, 650),
    framestyle = :box,
    grid = false,
    legend =:none,
    marker = :none
)

savefig("survival_mixed.svg")


"/Users/anna/Documents/megafauna_evolution/survival_mixed.svg"

In [ ]:
using StatsPlots, Colors

mass = 1510
h = 1
tmax = 30
f_bins = 50

S_slice = T[mass][h, :, :]
S_slice = round.(Int, S_slice)

# Mapear valores para índices de cor
color_mapping = Dict(0 => RGB(0,0,0), 1 => RGB(0.5,0,0.5), 2 => RGB(0,0.75,1))
S_colored = [color_mapping[val] for val in S_slice]

p = heatmap(
    1:tmax,
    1:f_bins,
    S_slice,
    xlabel = "Time (days)",
    ylabel = "Fat state (kcal)",
    title = "Habitat choice over time – Browsers",
    c = cgrad([:black, :purple, :deepskyblue], 3, categorical=true, rev=false),
    colorbar = :none,
    size = (800, 510),
    dpi = 300,
    right_margin = 5Plots.mm  # <-- adiciona margem à direita
)

scatter!([], [], label="Open Habitat", marker=:square, ms=10, color=:purple,
         legend=:outerright, legendfontsize=10)
scatter!([], [], label="Closed Habitat", marker=:square, ms=10, color=:deepskyblue)
savefig("sdp_browser.png")

In [ ]:
# Escolha do patch e tempo para visualizar
mass=1500
h = 1 # Patch selecionado
tmax = 30
f_bins =  50

# Plotando S para um patch específico ao longo de y e t
S_slice = T[mass][h, :, :]
S_slice_transposed = S_slice'

plotss = StatsPlots.heatmap(1:tmax, 1:f_bins, S_slice,
        xlabel="Time",
        ylabel="Fat state",
        title="Browsers",
        colorbar_title="Survival",
        c=:inferno)
#plot(plotss)

In [ ]:
#!/usr/bin/env julia

using CSV
using DataFrames
using StatsPlots
using Plots

# ============================================================
# Carregar dados
# ============================================================
script_dir = @__DIR__
data_file = joinpath(script_dir, "data_2", "Table_C2_Linxia_Basin_Body_Masses.csv")

println("Carregando dados: ", data_file)
df = CSV.read(data_file, DataFrame)
println("📋 Colunas disponíveis: ", names(df))

# ============================================================
# Selecionar colunas de interesse
# ============================================================
target_periods = ["Miocene", "Pliocene"]

for col in target_periods
    if !(col in names(df))
        error("A coluna '$col' não foi encontrada. Colunas disponíveis: $(names(df))")
    end
end

# ============================================================
# Combinar todas as massas (todas as colunas exceto NA)
# ============================================================
all_masses = Float64[]
for c in names(df)
    vals = df[!, c]
    vals = skipmissing(vals)
    append!(all_masses, filter(isfinite, collect(vals)))
end

# Converter de gramas → kg
all_masses_kg = all_masses ./ 1000
log_all_masses = log10.(all_masses_kg)

# ============================================================
# Configuração global de estilo
# ============================================================
default(
    fontfamily = "Helvetica",
    guidefont = font(18),
    tickfont = font(15),
    legendfont = font(13),
    dpi = 600
)

# ============================================================
# Define os ticks EXATAMENTE como em plot_mean_fitness.jl
# ============================================================
xticks_values = [0, 1, 2, 3, 4]  # eixo está em log10(massa)
xticks_labels = ["10⁰", "10¹", "10²", "10³", "10⁴"]

# ============================================================
# Criar gráfico
# ============================================================
p = plot(
    xlabel = "Body mass (kg)",
    ylabel = "Density",
    legend = :bottom,
    legend_background_color = :transparent,
    legend_foreground_color = :transparent,
    grid = false,
    framestyle = :box,
    size = (800, 650),
    xticks = (xticks_values, xticks_labels),   # ← EXATAMENTE como em plot_mean_fitness.jl
    xlim = (-0.2, 4.2)  # Adiciona espaçamento nas bordas
)

# 🎨 Cores distintas
colors = Dict(
    "All periods" => "#2b565e",   
    "Miocene" => "#843505",
    "Pliocene" => "#f8a916"
)

# ============================================================
# Curva geral (todas as massas)
# ============================================================
density!(
    p,
    log_all_masses,
    lw = 5,
    color = colors["All periods"],
    label = "Oligocene to Pleistocene",
    fill = false
)

# ============================================================
# Curvas específicas — Miocene e Pliocene
# ============================================================
for col in target_periods
    masses = df[!, col]
    masses = collect(skipmissing(masses))
    masses = filter(isfinite, masses)
    masses_kg = masses ./ 1000
    log_masses = log10.(masses_kg)

    if !isempty(log_masses)
        density!(
            p,
            log_masses,
            lw = 5,
            color = colors[col],
            label = col,
            fill = false
        )
    end
end

# ============================================================
# Salvar figura
# ============================================================
results_dir = joinpath(script_dir, "results")
isdir(results_dir) || mkpath(results_dir)

output_file = joinpath(results_dir, "mass_density_all_miocene_pliocene.svg")
savefig(p, output_file)
plot(p)
#println("✅ Curva de densidade (All + Miocene + Pliocene) salva em: $output_file")